**Part 3 · LangGraph** · step 18 of 25  
[← 17 State & reducers](17_state_and_reducers.ipynb) · [🏠 Roadmap](../README.md) · [19 Research chatbot →](19_research_chatbot_multiple_tools.ipynb)

# 18 · Tool-Calling Agent as a Graph (ReAct loop)

Here we build by hand what `create_agent` (notebook 10) gives you for free:

```
START → llm_agent ──tools_condition──► tools ─┐
           ▲                                  │
           └──────────────────────────────────┘
           └──(no tool calls)──► END
```

| Prebuilt | Purpose |
|---|---|
| `ToolNode(tools)` | Executes every tool call in the last `AIMessage` and returns `ToolMessage`s |
| `tools_condition` | Router: returns `"tools"` if the last message has tool calls, else `END` |

In [ ]:
from dotenv import load_dotenv

# Reads API keys (OPENAI_API_KEY, GROQ_API_KEY, GOOGLE_API_KEY, TAVILY_API_KEY)
# from the .env file at the AgenticAI root. load_dotenv() walks up parent
# folders until it finds one, so every notebook can share a single .env.
load_dotenv()

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict

from langchain.chat_models import init_chat_model
from langchain.messages import AnyMessage, HumanMessage
from langchain.tools import tool
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition

## Tools and an LLM that knows about them

In [ ]:
@tool
def add_two_number(a: int, b: int) -> int:
    """Add two numbers and return the result."""
    return a + b


@tool
def subtract_two_number(a: int, b: int) -> int:
    """Subtract b from a and return the result."""
    return a - b


@tool
def multiply_two_number(a: int, b: int) -> int:
    """Multiply two numbers and return the result."""
    return a * b


input_tools = [add_two_number, subtract_two_number, multiply_two_number]
llm_with_tools = init_chat_model("gpt-4o").bind_tools(input_tools)

## Build the graph

In [ ]:
class State(TypedDict):
    """Chat history; add_messages appends new messages."""

    messages: Annotated[list[AnyMessage], add_messages]


def llm_agent(state: State) -> dict:
    """Call the LLM with the full history and return ONLY its new message."""
    return {"messages": [llm_with_tools.invoke(state["messages"])]}


builder = StateGraph(State)
builder.add_node("llm_agent", llm_agent)
builder.add_node("tools", ToolNode(input_tools))

builder.add_edge(START, "llm_agent")
builder.add_conditional_edges("llm_agent", tools_condition)  # -> "tools" or END
builder.add_edge("tools", "llm_agent")                       # loop back after tools run

graph = builder.compile()

In [ ]:
from IPython.display import Image, display


def show_graph(compiled_graph):
    """Render a compiled graph as a PNG.

    draw_mermaid_png() uses the mermaid.ink web service; if you are offline
    it falls back to printing the Mermaid source (paste it into mermaid.live).
    """
    try:
        display(Image(compiled_graph.get_graph().draw_mermaid_png()))
    except Exception:
        print(compiled_graph.get_graph().draw_mermaid())


    show_graph(graph)

In [ ]:
result = graph.invoke({"messages": [HumanMessage("add 2 + 2 and 2 * 2 and 2+8*2+2")]})
for m in result["messages"]:
    m.pretty_print()

### ⚠️ Mistakes made (and fixed) while learning this
1. **Extra `add_edge("llm_agent", END)`** next to `add_conditional_edges("llm_agent", tools_condition)`.
   `tools_condition` already routes to `END`; a second static edge makes *both* paths fire.
2. **Returning the whole list** (`state["messages"] + [reply]`) — with `add_messages` return only the new message.
3. **Reading a key that may be missing** (`state["count"]`) — use `state.get("count", 0)`.

## Extra state keys — a call counter
State can hold more than messages. Keys without a reducer are overwritten by each update.

In [ ]:
class CounterState(TypedDict):
    """Messages plus how many times the LLM node ran."""

    count: int
    messages: Annotated[list[AnyMessage], add_messages]


def counting_llm(state: CounterState) -> dict:
    """Call the LLM and increment the counter."""
    return {
        "count": state.get("count", 0) + 1,
        "messages": [llm_with_tools.invoke(state["messages"])],
    }


workflow = StateGraph(CounterState)
workflow.add_node("llm_agent", counting_llm)
workflow.add_node("tools", ToolNode(input_tools))
workflow.add_edge(START, "llm_agent")
workflow.add_conditional_edges("llm_agent", tools_condition)
workflow.add_edge("tools", "llm_agent")
counter_graph = workflow.compile()

resp = counter_graph.invoke({"count": 0, "messages": [HumanMessage("add 2 + 2, then multiply the result by 10")]})
print("LLM calls:", resp["count"])
resp["messages"][-1].content

## Memory — why "what is my name?" failed
Each `invoke` without a checkpointer starts from an empty state, so the graph forgets everything.
Compile with a **checkpointer** and pass a `thread_id` (same idea as agents in notebook 11).

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

memory_graph = workflow.compile(checkpointer=InMemorySaver())
config = {"configurable": {"thread_id": "ujjawal-1"}}

memory_graph.invoke({"messages": [HumanMessage("my name is Ujjawal Kant and I like coffee")]}, config)
resp = memory_graph.invoke({"messages": [HumanMessage("what is my name?")]}, config)
resp["messages"][-1].content

---
✅ **Done with 18 Tool-calling agent graph.** Next: [19 Research chatbot →](19_research_chatbot_multiple_tools.ipynb)  
[🏠 Roadmap](../README.md)